# 07 - Laboratorio Hive Municipal

Este notebook demuestra como usar HDFS, Hive Metastore y Spark SQL sobre los Parquet Gold/Silver del proyecto municipal.

## Arquitectura

Raw -> Bronze Parquet -> Silver Parquet limpio -> Gold dimensional -> HDFS -> Hive external tables -> Power BI.

In [ ]:
from pyspark.sql import SparkSession
import os

builder = SparkSession.builder.appName('notebook-hive-municipal')
if os.getenv('HIVE_METASTORE_URI'):
    builder = builder.config('hive.metastore.uris', os.getenv('HIVE_METASTORE_URI'))
spark = builder.enableHiveSupport().getOrCreate()
spark

## Publicar tablas externas

Ejecutar desde terminal del contenedor si aun no se publicaron las tablas:

```bash
python scripts/hive_bootstrap.py --layer all
```

In [ ]:
spark.sql('SHOW DATABASES').show(truncate=False)
spark.sql('SHOW TABLES IN municipal_gold').show(50, truncate=False)

## Consulta 1: Lectura de Parquet

In [ ]:
spark.sql('SELECT COUNT(*) AS filas FROM municipal_gold.fact_ingresos_mensuales').show()

## Consulta 2: Agregacion anual

In [ ]:
spark.sql('''
SELECT year, SUM(MONTO_PIA) pia, SUM(MONTO_PIM) pim, SUM(MONTO_RECAUDADO) recaudado
FROM municipal_gold.fact_ingresos_mensuales
GROUP BY year
ORDER BY year
''').show(30, truncate=False)

## Consulta 3: Ranking con window function

In [ ]:
spark.sql('''
WITH ranking AS (
  SELECT
    m.DEPARTAMENTO_NOMBRE,
    m.DISTRITO_NOMBRE,
    SUM(i.MONTO_RECAUDADO) recaudado,
    RANK() OVER (PARTITION BY m.DEPARTAMENTO_NOMBRE ORDER BY SUM(i.MONTO_RECAUDADO) DESC) ranking_departamento
  FROM municipal_gold.fact_ingresos_mensuales i
  JOIN municipal_gold.dim_municipalidad_gold m ON i.SEC_EJEC = m.SEC_EJEC
  WHERE i.year = 2024
  GROUP BY m.DEPARTAMENTO_NOMBRE, m.DISTRITO_NOMBRE
)
SELECT * FROM ranking WHERE ranking_departamento <= 5
ORDER BY DEPARTAMENTO_NOMBRE, ranking_departamento
''').show(100, truncate=False)

## Vistas para los 6 dashboards

Las vistas se crean con `sql/hive/03_dashboard_views.sql` y se consumen desde Power BI.

In [ ]:
spark.sql("SHOW TABLES IN municipal_gold LIKE 'vw_dashboard_*'").show(20, truncate=False)